## Notebook12

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
food = (
    pl.read_csv(ub + "data/wweia_food.csv", ignore_errors=True)
    .select(
        c.id, c.day_of_week, c.time, c.meal_name, c.food_source,
        c.at_home, c.grams, c.kcal, c.protein, c.carbs, c.sugar, c.fat
    )
)

### Questions

This is What We Eat in America, the dietary part of a national health survey. A row is
one food item eaten by one person: 173,174 items reported by 11,925 people. Each person
was interviewed once and asked to recall everything they ate in the previous day, so a
person appears many times, once per item, and always on the same day.

The columns follow from that. `id` is the person, `day_of_week` is an integer from 1 to
7, and `time` is the hour of the day the item was eaten. `meal_name` is what the person
called the eating occasion and `food_source` is where the food came from. `grams` is the
mass of the item and `kcal` its energy in calories; `protein`, `carbs`, `sugar`, and
`fat` are the grams of each in that item.

The plots in this chapter do a grouped computation before they draw anything, which
means most of what you already know about `group_by` and `agg` still applies. It also
means the plots will happily draw a grouping you did not intend.

Unless a question says otherwise, just print the result of each block; do not save it to
a variable.

1. Start where the chapter starts, with the computation rather than the plot. Count the
items reported at each kind of meal and sort the counts from largest to smallest. There
are more meal names than fit in a default printout, so use `.print(20)` to see all of
them.

Dinner, lunch, breakfast, and snack are the four big ones, and together they are about
76 percent of the items. Below them the list gets stranger: `Drink`, `Supper`,
`Extended consumption`, and then ten categories that are not English words at all.

2. Look at the tail of that table. What are `Cena`, `Desayano`, `Almuerzo`, and
`Merienda` doing in a survey of American eating? Write down what you think happened.

**Answer**:
`Desayano` is breakfast. The survey was conducted in both English and Spanish, and the
interviewer wrote down the label the participant used, in the language the participant
used it in. Around 90 percent of the rows carrying a Spanish meal label come from an
interview conducted in Spanish, which is a claim we can check directly once we know how
to join this table to the table of participants. Note that the survey misspells its own
category: the Spanish word is *desayuno*.

This matters for every count we are about to make. A `group_by` on `meal_name` splits
dinner across `Dinner`, `Supper`, and `Cena`, so the four big categories are undercounts
of the meals they name. The column looks like a property of the food. Part of it is a
property of the interview.

3. Now hand the same column to a plot. `geom_bar` takes one categorical aesthetic,
counts the rows in each category, and draws each count as a bar, so it should reproduce
the table you just made. Flip the coordinates so the labels are readable.

The bar heights are right and the plot is close to useless. `Other` and `Tentempie`, the
two smallest categories in the data, sit at the top, and `Dinner` is at the bottom.
Nothing sorted them, which is the point: a categorical axis follows the order the values
first appear in the data, and `meal_name` first appears in whatever order the survey
happened to store its rows. There is no ordering step inside the plot to reach for.

4. Ordering the plot means ordering the data. There are two ways to do it, and they
differ in who does the counting. The first is to compute the counts yourself, sort them,
and hand the finished numbers to `geom_col`, which draws the `y` you give it and counts
nothing. The second is to leave the counting to `geom_bar` and sort the rows by a window
aggregate, so that every row of a category carries the size of its own category.

The two plots are identical, and in neither one did we tell the plot to sort anything.
The first sorts a table of 20 rows on a column that is right there. The second sorts a
table of 173,174 rows on a number that does not exist in it: `pl.len().over(c.meal_name)`
writes each category's size onto every row of that category. That is last chapter's
window function, earning its keep in a bar chart. Reach for `geom_col` when the number
you want is already in the frame, and for `geom_bar` when it is a plain count of rows and
you would rather not compute it.

5. Here is where the two geometries stop being interchangeable, because the number worth
drawing is not always a count. Draw the average calories per item by `food_source`. Drop
the rows where the source was not recorded, and keep only sources with at least 500
items, since a mean over a handful of rows is not a number anyone should plot.

Thirteen sources survive the filter. A fast food or pizza item averages 210 calories, a
convenience store item 182, and a supermarket item 133. The lightest sources are mail
order, at 76, and a child or adult care center, at 104. `geom_bar` cannot draw this plot
at all: it has one trick, and that trick is counting rows. A mean has to be computed
before it can be drawn.

Note also what the bars are not saying. The mean is per *item*, not per meal or per day,
so this is a statement about how calorically dense a single recorded item tends to be.
Fast food arrives in bigger units, and one of the reasons the supermarket bar is low is
that a glass of water from your own kitchen is an item too.

6. Turn to the calories themselves. A histogram cuts a numeric column into equal
intervals and counts the values in each one, which makes it a grouped count where the
groups are intervals. Draw one for `kcal` with a bin width of 25. Then cut the zeros and
the long right tail and draw it again.

**Answer**:
axis, and the tail runs out to a 4,862 calorie item around x = 5000. That first bin holds
about 55,000 items, and 23,132 of them have exactly zero calories. Most of those are
water: 11,069 bottled and 10,111 from the tap. Nearly half of everything filed under
`Drink` is a zero. The zeros are real data, but they are a different population from the
food, and while they are in the frame they set the y-axis and flatten everything else.

In the second plot the shape is visible: a peak in the smallest bin and a long, steady
decline. The median item is 79 calories. Be honest with yourself about what those two
filters did, though. They did not fix the plot; they chose a subset of the data and
plotted that. The 23,132 zeros and the 4,862 calorie item are still out there, and a
reader looking at the second histogram cannot see either one.

7. Items are not the unit anyone actually cares about. Add up each person's day and look
at the distribution of daily totals instead. Save this one as `person`, since we use it
again below.

A single humped curve, peaking around 1,600 calories and skewed to the right. The median
person reports 1,831 calories for the day; the largest total in the data is 12,501. The
`.over()` you might have reached for here is not needed, because summing a person's day
is a real aggregation: it collapses 173,174 rows to 11,925, one per person, and that is
the table the picture is about.

There is a well-known problem with that median. It is lower than what American adults
actually eat, and the reason is the method: this is what people *said* they ate when
asked to remember the previous day. Under-reporting in dietary recall is large,
systematic, and not evenly spread across people. Every number in this notebook inherits
it.

8. A boxplot summarizes a numeric column separately for each level of a categorical one.
Total up what each person ate at each of the four main meals, then draw the four
distributions, ordered by their medians.

The medians run 347 calories at breakfast, 360 at snacks, 510 at lunch, and 639 at
dinner. The ordering trick is the one from question 4 with a median in place of a count:
`.sort(c.kcal.median().over(c.meal_name))` writes each meal's median onto every row of
that meal and sorts on it. Sorting on a plain `.sort(c.kcal)` would have ordered the
meals by whichever person in each one happened to land first.

Snacks are the interesting box. The median is near breakfast's, but the box is much
wider and the whisker runs further, because "snack" covers both a stick of gum and a
second dinner.

9. Now the same idea on the day of the week. Draw a boxplot of each person's daily
calories against `day_of_week`, which is stored as an integer. Then cast the column to a
string, sorting the rows first, and draw it again.

**Answer**:
Plotnine reads a column of numbers as a continuous quantity, so it saw no categories to
split on and drew a single box over all 11,925 people, parked at the midpoint of the
range. Nothing raised an error, and the box it drew is a perfectly accurate summary of a
table you never asked it to summarize. Once the column is text, each distinct value
becomes a category and you get seven boxes.

The sort has to come before the cast. Sort a string column and you get 1, 2, 3, 4, 5, 6,
7 by luck here, but 1, 10, 11, 2 the moment there are more than nine categories. The
medians barely move, from 1,744 on day 1 to 1,898 on day 6.

The boxes are also not what they look like. They appear to be seven comparable groups,
and they are not: day 1 holds 942 people, while days 5, 6, and 7 hold 3,149, 2,797, and
2,005. That imbalance is a fact about when the survey scheduled its interviews, not about
when America eats, and no plot of this column will tell you which day is which.

10. Close with two numeric columns. `geom_smooth` fits a model to the relationship
between them and draws the fitted curve; with `method="lm"` it is a straight line, and
`se=False` turns off the confidence band. Plot each person's total mass of food against
their total calories, with the points behind the line.

The line rises from around 1,000 calories at the left edge to about 7,500 at the far
right, and the cloud around it is enormous: the correlation is 0.54, which for a
relationship this obvious is remarkably weak. Mass is not energy. The 39 people out past
10 kilograms are not eating ten kilograms of food; more than half of the mass they
reported came from items with zero calories, and most of it is logged under
`Extended consumption`, which is the survey's word for sipping something all day.

Two things to take from the picture. The `alpha=0.05` is doing real work; at full opacity
the 11,925 points are a black wall and you cannot see where the bulk of people actually
sit. And the left end of that line, sitting at roughly 1,000 calories, describes a person
who ate no food at all. The lightest day in the data is 55 grams. A fitted line will
extend itself anywhere you let it, and it has no idea when it has left the data behind.